In [24]:
IMAGE_RESOLUTION = 10

In [25]:
import geopandas as gpd
import numpy as np  

Importing polygons and cleaning

In [26]:
geopackage = gpd.read_file(r"C:\Users\fulle\Desktop\SEZPolygonML\data\SEZ POLYGONS.gpkg")
geopackage.head()

,id,Name,description,timestamp,begin,end,altitudeMode,tessellate,extrude,visibility,...,sez_id,name2,source_url_1,Notes,source_url_2,source_url_3,Confidence,Time_serching,Date_completion,geometry
0,None,Gift SEZ Limited,sez_id: GJ-0015<br>name: Gift SEZ Limited<br>s...,NaT,NaT,NaT,None,-1,0,-1,...,GJ-0015,None,http://en.wikipedia.org/wiki/File:GIFT_City_Ma...,,,,High,,,"MULTIPOLYGON Z (((72.67548 23.16274 0, 72.6743..."
1,None,M/S.GIGAPLEX ESTATE PVT.LTD.,sez_id: MH-0038<br>name: M/S.GIGAPLEX ESTATE P...,NaT,NaT,NaT,None,-1,0,-1,...,MH-0038,None,https://gis.midcindia.org/MIDCGISPortal/,Plot Status:\tAllotted\nPlot No\tIT-5\nRO NAME...,,,High,,,"MULTIPOLYGON Z (((72.9973 19.17193 0, 72.9973 ..."
2,None,iGate Global Solutions Ltd.,sez_id: MH-0040<br>name: iGate Global Solution...,NaT,NaT,NaT,None,-1,0,-1,...,MH-0040,None,https://gis.midcindia.org/MIDCGISPortal/,"Plot Status:\tnull\nPlot No\tIT-3,IT-4\nRO COD...",,,High,,,"MULTIPOLYGON Z (((72.99116 19.17732 0, 72.9889..."
3,None,Serene Properties Pvt. Ltd.,sez_id: MH-0016<br>name: Serene Properties Pvt...,NaT,NaT,NaT,None,-1,0,-1,...,MH-0016,None,https://gis.midcindia.org/MIDCGISPortal/,Plot Status:\tnull\nPlot No\t3\nRO CODE:\t\nIA...,,,High,,,"MULTIPOLYGON Z (((73.00024 19.15908 0, 73.0010..."
4,None,HGP Community Pvt. Ltd. (Formerly\nHiranandani...,sez_id: MH-0007<br>name: HGP Community Pvt. Lt...,NaT,NaT,NaT,None,-1,0,-1,...,MH-0007,None,https://brookfieldindiareit.in/files/results/D...,Hiranandani Kensington SEZ Building A and B,,,High,,,"MULTIPOLYGON Z (((72.91001 19.11125 0, 72.9101..."


In [27]:
(~geopackage["Confidence"].isin(["High", "Medium", "Low"])).sum()


np.int64(2)

In [28]:
geopackage[~geopackage["Confidence"].isin(["High", "Medium", "Low"])]

,id,Name,description,timestamp,begin,end,altitudeMode,tessellate,extrude,visibility,...,sez_id,name2,source_url_1,Notes,source_url_2,source_url_3,Confidence,Time_serching,Date_completion,geometry
206,None,LTIMindtree Limited (formerly Larsen & Toubro ...,sez_id: MH-0051<br>name: LTIMindtree Limited (...,NaT,NaT,NaT,None,-1,0,-1,...,MH-0051,None,,"Followed road, approx. area match. Relocated p...",,,Meduim,25,7/14,"MULTIPOLYGON Z (((73.03019 19.1104 0, 73.03053..."
218,None,L&T Phoenix Infoparks Private Limited\n,sez_id: TG-0006<br>name: L&T Phoenix Infoparks...,NaT,NaT,NaT,None,-1,0,-1,...,TG-0006,None,,"Surrounded all office builldings, good area match",,,Meduim,25,7/17,"MULTIPOLYGON Z (((78.3732 17.44806 0, 78.37269..."


In [29]:
geopackage.loc[~geopackage["Confidence"].isin(["High", "Medium", "Low"]), "Confidence"] = "Medium"
(~geopackage["Confidence"].isin(["High", "Medium", "Low"])).sum()

np.int64(0)

Need to "flatten" coordinate-based polygons to India-specific Lambert Conformal Conic projection EPSG:7755

In [30]:
LCCPolygons = geopackage.to_crs("EPSG:7755")

In [31]:
bounds = LCCPolygons.bounds
x_diffs = (bounds["maxx"] - bounds["minx"])
y_diffs = (bounds["maxy"] - bounds["miny"])
print(x_diffs.describe())
print(y_diffs.describe())

count      258.000000
mean      1147.869993
std       1981.052492
min        106.135624
25%        437.475232
50%        664.134001
75%       1271.767506
max      26559.619074
dtype: float64
count      258.000000
mean      1042.764763
std       1309.345180
min        102.150082
25%        412.265508
50%        605.114693
75%       1130.133752
max      12559.100639
dtype: float64


Need tiles to be at least:
26550.62 / 10 = 2655.062 pixels wide 
12559.1 / 10 = 1255.91 pixels tall
for 10m resolution.

In [33]:
minXPixels= x_diffs.max() / IMAGE_RESOLUTION
minYPixels = y_diffs.max() / IMAGE_RESOLUTION